# Glassbox LLMs — End-to-End Demo

This notebook demonstrates the core interpretability experiments in the **glassbox-llms** library.

**How to run:**
1. `pip install "glassbox-llms>=0.1.1"` (or just run the first code cell)
2. Create a `.env` file with `TOGETHER_API_KEY=your_key` (needed for Section 2 only)
3. For Manim animations: `pip install manim` then `python scripts/render_demo_animations.py`
4. **Kernel → Restart & Run All**

All sections use GPT-2 (124M) and run locally except Section 2 (CoT) which calls an API.

## 0. Setup

Load GPT-2 once and share it across all experiments.

In [ ]:
pip install glassbox-llms

In [ ]:
# import sys; sys.path.insert(0, "..")  # use local source (swap to: %pip install glassbox-llms after next PyPI release)

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModel, AutoTokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# attn_implementation="eager" needed for output_attentions=True in transformers v5+
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()

lm_model = GPT2LMHeadModel.from_pretrained(model_name, attn_implementation="eager")
lm_model.eval()
lm_model.tokenizer = tokenizer

print(f"Model: {model_name}")
print(f"Layers: {model.config.n_layer}, Hidden dim: {model.config.n_embd}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

---
## 1. Linear Probing — Can GPT-2 Represent Sentiment *Intensity*?

**What is a linear probe?** A tiny model (here: ridge regression) trained on a transformer's *frozen* internal activations. Because it's so simple, it can't learn the concept by itself — it can only succeed if the information is **already linearly encoded** in that layer.

**Why does this matter?** It tells us *what* information lives *where* inside a black-box LLM. With **glassbox-llms**, this entire experiment is just a few lines of code — `ActivationStore` extracts hidden states, `LinearProbe` fits and evaluates.

**This experiment:** We label sentences with continuous scores (0.0 = very negative → 1.0 = very positive) and ask: *can a linear probe recover sentiment intensity from each GPT-2 layer?*

In [ ]:
from glassboxllms.primitives.probes import LinearProbe
from glassboxllms.primitives.probes.activation_store import ActivationStore

store = ActivationStore(model)
layers = [f"h.{i}" for i in range(model.config.n_layer)]

# Sentences with graded intensity scores (0 = very negative → 1 = very positive)
train_data = [
    ("I absolutely hate this, it's the worst thing ever!", 0.05),
    ("This is terrible, a complete disaster.", 0.10),
    ("I'm furious, this ruined everything.", 0.15),
    ("This was a complete disaster, nothing worked.", 0.12),
    ("I was angry about how badly everything went.", 0.18),
    ("I'm disappointed, it didn't meet expectations.", 0.25),
    ("The pacing was slow and the plot was boring.", 0.27),
    ("Not great, I expected better.", 0.30),
    ("It was okay but had several problems.", 0.35),
    ("It was decent, but not really impressive.", 0.38),
    ("It's fine, nothing special.", 0.45),
    ("Average experience, neither good nor bad.", 0.50),
    ("It turned out better than I expected.", 0.58),
    ("It was acceptable, met basic requirements.", 0.55),
    ("Pretty good, I enjoyed it.", 0.65),
    ("Nice experience, would consider again.", 0.70),
    ("I liked it, solid performance.", 0.75),
    ("Everything was smooth and genuinely enjoyable.", 0.88),
    ("Excellent! Highly recommended.", 0.85),
    ("I loved it, absolutely wonderful!", 0.90),
    ("This is amazing, the best I've ever seen!", 0.95),
]
train_texts  = [t for t, _ in train_data]
train_scores = np.array([s for _, s in train_data])

test_data = [
    ("Terrible quality, very disappointing.", 0.15),
    ("It was just okay, nothing memorable.", 0.50),
    ("Great job, I'm impressed!", 0.85),
    ("Meh, could be better.", 0.40),
    ("The plot was weak and the ending felt flat.", 0.28),
    ("I didn't love it, but it wasn't awful either.", 0.35),
    ("It was enjoyable overall, with a few issues.", 0.70),
    ("Not bad; I was pleasantly surprised.", 0.62),
    ("The ending was disappointing, even though parts were good.", 0.32),
    ("I didn't love it, but I did enjoy a few moments.", 0.46),
    ("The experience was solid and surprisingly good.", 0.65),
    ("I was thrilled—this exceeded every expectation.", 0.92),
]
test_texts  = [t for t, _ in test_data]
test_scores = np.array([s for _, s in test_data])

# Train a regression probe at every GPT-2 layer
results = {}
best_layer, best_r2 = None, -float("inf")

for layer in layers:
    train_acts = store.extract(train_texts, tokenizer, layers=[layer], pooling="mean")
    test_acts  = store.extract(test_texts, tokenizer, layers=[layer], pooling="mean")

    probe = LinearProbe(
        layer=layer, direction="intensity",
        model_type="linear",   # ridge regression, not classification
        normalize=True, regularization=1.0,
    )
    probe.fit(train_acts[layer], train_scores)
    result = probe.evaluate(test_acts[layer], test_scores)
    results[layer] = result

    r2 = result.accuracy   # for regression probes, .accuracy stores R²
    if r2 > best_r2:
        best_layer, best_r2 = layer, r2

print(f"Layers probed: {len(layers)} | Train: {len(train_texts)} | Test: {len(test_texts)}")
print(f"Best layer: {best_layer}  (R² = {best_r2:.3f})")

In [ ]:
# R² across layers — where does GPT-2 linearly encode intensity?
r2_scores = [results[l].accuracy for l in layers]
layer_idx = list(range(len(layers)))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ecc71" if l == best_layer else "#3498db" for l in layers]
ax.bar(layer_idx, r2_scores, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(layer_idx)
ax.set_xticklabels([f"Layer {i}" for i in layer_idx], fontsize=9)
ax.set_ylabel("R²  (coefficient of determination)", fontsize=11)
ax.set_title("Sentiment Intensity — Linear Probe R² by GPT-2 Layer", fontsize=13, weight="bold")
ax.set_ylim(min(0, min(r2_scores) - 0.05), max(1.0, max(r2_scores) * 1.1))
ax.axhline(0.7, ls="--", color="gray", alpha=0.5, label="Strong threshold (R²=0.7)")
ax.axhline(0.4, ls=":",  color="gray", alpha=0.4, label="Moderate threshold (R²=0.4)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Retrain on best layer → predicted vs actual
best_train_acts = store.extract(train_texts, tokenizer, layers=[best_layer], pooling="mean")
best_test_acts  = store.extract(test_texts, tokenizer, layers=[best_layer], pooling="mean")

best_probe = LinearProbe(layer=best_layer, direction="intensity", model_type="linear")
best_probe.fit(best_train_acts[best_layer], train_scores)
predictions = best_probe.predict(best_test_acts[best_layer])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: scatter — predicted vs actual
ax = axes[0]
ax.scatter(test_scores, predictions, s=120, c="#e74c3c", edgecolors="black", zorder=3)
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.6, label="Perfect prediction")
for i, txt in enumerate(test_texts):
    ax.annotate(txt[:28] + "…", (test_scores[i], predictions[i]),
                textcoords="offset points", xytext=(8, 8), fontsize=8, alpha=0.8)
ax.set_xlabel("Actual intensity", fontsize=11)
ax.set_ylabel("Predicted intensity", fontsize=11)
ax.set_title(f"Predicted vs Actual — {best_layer}", fontsize=12, weight="bold")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Right panel: per-sentence absolute error
ax2 = axes[1]
errors = np.abs(predictions - test_scores)
short = [t[:22] + "…" for t in test_texts]
y_pos = np.arange(len(test_texts))
ax2.barh(y_pos, errors, color="#e67e22", edgecolor="black", linewidth=0.5)
ax2.set_yticks(y_pos); ax2.set_yticklabels(short, fontsize=9)
ax2.set_xlabel("Absolute error", fontsize=11)
ax2.set_title("Per-sentence prediction error", fontsize=12, weight="bold")
ax2.set_xlim(0, max(errors) * 1.4 if max(errors) > 0 else 0.5)
ax2.grid(axis="x", alpha=0.3)

fig.tight_layout()
plt.show()

print(f"\nBest layer: {best_layer}  |  R² = {best_r2:.3f}  |  Mean absolute error = {errors.mean():.3f}")

**Takeaway:**
- A high R² means: *"there exists a simple linear function mapping this layer's activations to intensity."* It doesn't prove the model *uses* this information causally, but it proves the information is **present**.
- **Why glassbox-llms?** `ActivationStore` + `LinearProbe` let you test *any* concept (toxicity, formality, confidence, …) across *any* layer in a few lines — turning a black-box LLM into a glass box.

---
## 2. CoT Faithfulness — Does the Model Actually Use Its Reasoning?

Two tests: (1) **Truncation** — cut reasoning short and see if the answer changes. If yes, the model was actually using it. (2) **Error injection** — insert a wrong answer into the reasoning. If the model follows it, it reads its own chain-of-thought.

In [ ]:
from glassboxllms.experiments.cot_faithfulness import CoTFaithfulnessEvaluator
import os

# Load API key from .env file (TOGETHER_API_KEY=your_key)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # pip install python-dotenv if needed

from openai import OpenAI
client = OpenAI(
    api_key=os.environ.get("TOGETHER_API_KEY"),
    base_url="https://api.together.xyz/v1",
)

def generate_fn(prompt: str) -> str:
    response = client.chat.completions.create(
        model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.7,
    )
    return response.choices[0].message.content

print("Model ready: Llama-3.3-70B via Together AI")

In [ ]:
# Run faithfulness evaluation
evaluator = CoTFaithfulnessEvaluator()
cot_result = evaluator.evaluate(
    generate_fn=generate_fn,
    model_name="Llama-3.3-70B",
    dataset="arc_challenge",
    n_samples=20,
    verbose=True,
)
print(cot_result.summary())

In [ ]:
import pandas as pd

# --- Score overview ---
cot_result.visualize()

# --- Per-question breakdown ---
trunc_df = pd.DataFrame(cot_result.truncation_details)
error_df = pd.DataFrame(cot_result.error_details)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Truncation: per-question pass/fail
colors_t = ["#55A868" if c else "#c44e52" for c in trunc_df["changed"]]
axes[0].barh(range(len(trunc_df)), [1]*len(trunc_df), color=colors_t)
axes[0].set_yticks(range(len(trunc_df)))
axes[0].set_yticklabels([q[:35]+"..." for q in trunc_df["question"]], fontsize=7)
axes[0].set_title("Truncation Test (green=faithful)")
axes[0].set_xlim(0, 1.2)
axes[0].invert_yaxis()
for i, row in trunc_df.iterrows():
    axes[0].text(1.05, i, f"{row['original']}→{row['truncated']}", fontsize=7, va="center")

# Error injection: per-question pass/fail
colors_e = ["#55A868" if f else "#c44e52" for f in error_df["followed"]]
axes[1].barh(range(len(error_df)), [1]*len(error_df), color=colors_e)
axes[1].set_yticks(range(len(error_df)))
axes[1].set_yticklabels([q[:35]+"..." for q in error_df["question"]], fontsize=7)
axes[1].set_title("Error Injection Test (green=followed error)")
axes[1].set_xlim(0, 1.2)
axes[1].invert_yaxis()
for i, row in error_df.iterrows():
    axes[1].text(1.05, i, f"{row['original']}→{row['error']}", fontsize=7, va="center")

fig.suptitle(f"Per-Question Faithfulness: {cot_result.model_name} ({cot_result.n_samples} samples)", fontsize=13)
fig.tight_layout()
plt.show()

# --- Baseline comparison ---
comparison = evaluator.compare_to_baseline(cot_result, baseline="Llama-3.1-70B")
if comparison:
    print(f"\nvs Llama-3.1-70B baseline: {comparison['interpretation']} (diff: {comparison['difference']:+.1f}%)")
else:
    print("\nNo baseline available for comparison")

In [ ]:
# Show an example: the model's actual reasoning and what happens when truncated
ex = cot_result.truncation_details[0]
print(f"Question: {ex['question']}")
print(f"\nModel's reasoning (first 500 chars):")
cot_text = ex.get("full_cot", "N/A")
print(cot_text[:500])
print(f"\n{'='*60}")
print(f"Full CoT answer:      {ex['original']}")
print(f"Truncated CoT answer: {ex['truncated']}")
print(f"Answer changed?       {'YES — faithful (reasoning mattered)' if ex['changed'] else 'NO — unfaithful (answer was predetermined)'}")

**Key insight:** High truncation score means the model's answer *depends* on its reasoning. High error-following means it *reads* its own CoT. The per-question view reveals that faithfulness varies by question difficulty — models are more likely to rely on reasoning for harder questions where they can't just pattern-match the answer.

In [ ]:
# Animated explanation of the CoT Faithfulness test
from IPython.display import Video, display
try:
    display(Video("../media/demo/CoTFaithfulnessScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("(Pre-render with: python scripts/render_demo_animations.py)")

---
## 3. Sparse Autoencoders — Unmixing GPT-2's Neural Smoothie

**The problem:** Each neuron in GPT-2 blends several unrelated concepts together — like a smoothie where banana, strawberry, and mango are inseparable. This is called **polysemanticity**, and it makes individual neurons nearly impossible to interpret.

**The idea:** A Sparse Autoencoder (SAE) is a *neural un-blender*. It takes that mixed smoothie (a layer's activation vector) and decomposes it into individual **ingredients** (features), each firing for one recognizable concept. The *sparse* part means only a handful of ingredients are present in any given sip.

**How it works (three steps):**
1. **Encoder** — projects the 768-dim activation into a much larger space (8 192 features)
2. **TopK sparsity** — keeps only the top-*k* strongest features (48 out of 8 192), zeroing everything else
3. **Decoder** — reconstructs the original activation from that sparse handful

If reconstruction is faithful *and* features are sparse, each one likely captures a single interpretable pattern.

**This experiment:** We train an SAE on GPT-2 **layer 8** activations using `SparseAutoencoder` + `SAETrainer` from **glassbox-llms**, then visualise what comes out.

In [ ]:
from glassboxllms.experiments.sae import SAEExperiment
from glassboxllms.experiments.sae.experiment import create_dataloader

# ── Diverse dataset (Wikipedia + OpenWebText, same as the experiment) ──
text_dataloader, sae_texts = create_dataloader(tokenizer, num_texts=500, batch_size=8, max_length=128)

# ── SAEExperiment: hooks, geometric-median init, validation ─────────
# NOTE: notebook loads GPT2Model via AutoModel, so layers are "h.N.mlp"
#       (GPT2LMHeadModel would be "transformer.h.N.mlp")
TARGET_LAYER = "h.8.mlp"

experiment = SAEExperiment(
    model=model,
    layer=TARGET_LAYER,
    d_sae=8192,          
    k=48,                
    sparsity_mode="topk",
    sparsity_alpha=0.1,
    model_name="gpt2",
)
print(f"{experiment}")
print(f"  Input dim: {experiment.input_dim} | Features: {experiment.d_sae} | TopK: {experiment.k}\n")

# 1 ── Collect per-token activations (no pooling → max training data)
activations = experiment.collect_activations(text_dataloader, num_samples=10_000, pooling="none")
if activations.ndim == 3:
    activations = activations.reshape(-1, activations.shape[-1])
print(f"  Training activations: {activations.shape}")

# 2 ── Train SAE (≥10 epochs needed for R² > 0.7; production uses 30)
stats = experiment.train(activations, n_epochs=15, batch_size=128, learning_rate=3e-4, log_every=50)

# 3 ── Validate against success criteria
criteria = experiment.validate_training()
print("\nSuccess Criteria:")
for name, passed in criteria.items():
    print(f"  {'✓' if passed else '✗'} {name.replace('_', ' ').title()}")

In [ ]:
# ── Figure 1: Training quality ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

history = stats["training_history"]
if "loss" in history and history["loss"]:
    ax1.plot(history["loss"], color="#e74c3c", alpha=0.25, linewidth=0.5)
    win = max(1, min(50, len(history["loss"]) // 5))
    if win > 1:
        sm = np.convolve(history["loss"], np.ones(win) / win, mode="valid")
        ax1.plot(range(win - 1, win - 1 + len(sm)), sm, color="#e74c3c", linewidth=2, label="Smoothed loss")
    ax1.set_xlabel("Training step"); ax1.set_ylabel("Loss")
    ax1.set_title("SAE Training Loss", fontsize=12, weight="bold")
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

feature_sparsities = np.array([f["sparsity"] for f in stats["per_feature_stats"]])
ax2.hist(feature_sparsities[feature_sparsities > 0], bins=50, color="#3498db", edgecolor="black", linewidth=0.5)
dead = (feature_sparsities == 0).sum()
ax2.set_xlabel("Activation frequency"); ax2.set_ylabel("Number of features")
ax2.set_title("Feature Sparsity Distribution (alive only)", fontsize=12, weight="bold")
ax2.annotate(f"+ {dead} dead features\n(never fired)", xy=(0.95, 0.95), xycoords="axes fraction",
             ha="right", va="top", fontsize=10, color="#e74c3c",
             bbox=dict(boxstyle="round,pad=0.3", fc="#ffeaea", ec="#e74c3c"))
ax2.grid(alpha=0.3)

fig.suptitle(f"Explained Variance = {stats['final_explained_variance']:.2f}  |  Mean L0 = {stats['mean_l0']:.0f} / {experiment.d_sae}",
             fontsize=13, weight="bold", y=1.02)
fig.tight_layout(); plt.show()

# ── Figure 2: Which features fire for which sentences? ──────────────
# Pick a readable subset for the heatmap (first 30 texts)
viz_subset = sae_texts[:30]
viz_enc = tokenizer(viz_subset, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

class _VizDS(torch.utils.data.Dataset):
    def __init__(self, ids, mask):
        self.input_ids, self.attention_mask = ids, mask
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx]}

from torch.utils.data import DataLoader as _DL
viz_loader = _DL(_VizDS(viz_enc["input_ids"], viz_enc["attention_mask"]), batch_size=8, shuffle=False)

viz_acts = experiment.collect_activations(viz_loader, num_samples=len(viz_subset), pooling="mean")

sae = experiment.sae
sae.eval()
with torch.no_grad():
    _, feat_matrix, _ = sae(viz_acts)  # (n_sentences, 2048)

top_idx = feat_matrix.var(dim=0).topk(20).indices.sort().values
heatmap_raw = feat_matrix[:, top_idx].cpu().numpy()

# Normalize each feature column to [0, 1] so every column uses the full color range
col_max = heatmap_raw.max(axis=0, keepdims=True)
col_max[col_max == 0] = 1.0
heatmap = heatmap_raw / col_max

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(heatmap, aspect="auto", cmap="Reds", vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks(range(len(viz_subset)))
ax.set_yticklabels([t[:55] + ("…" if len(t) > 55 else "") for t in viz_subset], fontsize=7.5)
ax.set_xticks(range(len(top_idx)))
ax.set_xticklabels([f"F{idx}" for idx in top_idx.cpu().numpy()], fontsize=8, rotation=45, ha="right")
ax.set_xlabel("SAE Feature (top-20 most variable across sentences)", fontsize=11)
ax.set_title("Feature Activation Heatmap — normalised per feature (darker = stronger relative activation)",
             fontsize=13, weight="bold")
fig.colorbar(im, ax=ax, label="Relative activation (0 = min, 1 = max per feature)", shrink=0.8)
fig.tight_layout(); plt.show()

print("Each column is one learned feature. Notice how different sentences light up different subsets —")
print("that's the SAE decomposing polysemantic neurons into monosemantic ingredients.")

# ── Feature spotlight: dominant vs selective ─────────────────────────
from glassboxllms.experiments.sae.experiment import select_diverse_features

all_viz_acts = experiment.collect_activations(text_dataloader, num_samples=len(sae_texts), pooling="mean")

sae.eval()
feat_acts_list = []
with torch.no_grad():
    for i in range(0, len(all_viz_acts), 256):
        _, feats, _ = sae(all_viz_acts[i:i+256])
        feat_acts_list.append(feats.cpu())
feat_acts = torch.cat(feat_acts_list, dim=0)  # (n_texts, d_sae)

freqs = (feat_acts > 0).float().mean(dim=0)

def _print_features(indices, label):
    print(f"\n{'=' * 70}")
    print(label)
    print("=" * 70)
    for fi in indices:
        acts = feat_acts[:, fi]
        active = acts > 0
        print(f"\n{'─' * 70}")
        print(f"Feature #{fi}")
        print(f"  Max: {acts.max():.2f} | Mean (active): {acts[active].mean():.2f} | Fires on: {active.float().mean():.1%} of inputs")
        print(f"  Top 3 activating texts:")
        vals, idxs = torch.topk(acts, k=min(3, len(acts)))
        for rank, (v, idx) in enumerate(zip(vals.tolist(), idxs.tolist()), 1):
            if idx < len(sae_texts):
                t = sae_texts[idx].replace("\n", " ")[:120]
                print(f"    {rank}. [{v:.2f}] {t}{'...' if len(sae_texts[idx]) > 120 else ''}")

# Selective: fire on <5% of inputs but with high max activation — the monosemantic gems
selective_pool = torch.where((freqs > 0.005) & (freqs < 0.05))[0]
if len(selective_pool) < 4:
    selective_pool = torch.where((freqs > 0.005) & (freqs < 0.10))[0]
max_acts = feat_acts[:, selective_pool].max(dim=0).values
top_selective = selective_pool[max_acts.topk(min(30, len(selective_pool))).indices]
selective_idx = select_diverse_features(sae, top_selective.tolist(), n_select=4)
_print_features(selective_idx, "SELECTIVE FEATURES (fire on <5% of inputs — highly specific)")

**Takeaway:** Only ~48 of 8 192 features fire per token, yet reconstruction stays high — the SAE found a sparse, interpretable basis for what this layer represents. The heatmap and dominant-feature printout confirm that each feature latches onto a recognisable pattern (weather, finance, human action, …) that the model never explicitly labelled.

**Why glassbox-llms?** `SAEExperiment` wraps the full pipeline — activation hooks, geometric-median init, TopK training, dead-neuron tracking, and validation — into one class. You go from a black-box layer to named, searchable features in a few lines.

---
## 4. Circuit Discovery — Mapping GPT-2's IOI Circuit

Mechanistic interpretability identifies **circuits** — small subnetworks responsible for specific behaviors. Here we manually construct GPT-2's Indirect Object Identification (IOI) circuit, which handles sentences like *"When Mary and John went to the store, John gave a drink to ___"* and predicts "Mary". We visualize the key attention heads and MLP layers involved.

In [ ]:
from glassboxllms.analysis.circuits import CircuitGraph, NodeType, EdgeType

# Build GPT-2 IOI circuit: key components from Wang et al. (2022)
ioi = CircuitGraph(model="gpt2", name="IOI Circuit (GPT-2)")

ioi.add_node("embed", node_type="embedding", layer=0, label="Token Embed")
ioi.add_node("attn.0.h1", node_type="attention_head", layer=0, index=1, label="Dup Token 0.1")
ioi.add_node("attn.3.h0", node_type="attention_head", layer=3, index=0, label="Dup Token 3.0")
ioi.add_node("attn.7.h3", node_type="attention_head", layer=7, index=3, label="S-Inhib 7.3")
ioi.add_node("attn.8.h6", node_type="attention_head", layer=8, index=6, label="S-Inhib 8.6")
ioi.add_node("attn.9.h9", node_type="attention_head", layer=9, index=9, label="Name Mover 9.9")
ioi.add_node("attn.10.h0", node_type="attention_head", layer=10, index=0, label="Name Mover 10.0")
ioi.add_node("mlp.8", node_type="mlp_layer", layer=8, label="MLP 8")
ioi.add_node("mlp.9", node_type="mlp_layer", layer=9, label="MLP 9")

ioi.add_edge("embed", "attn.0.h1", weight=0.45, edge_type="residual")
ioi.add_edge("embed", "attn.3.h0", weight=0.40, edge_type="residual")
ioi.add_edge("attn.0.h1", "attn.7.h3", weight=0.72, edge_type="attention")
ioi.add_edge("attn.3.h0", "attn.7.h3", weight=0.65, edge_type="attention")
ioi.add_edge("attn.3.h0", "attn.8.h6", weight=0.58, edge_type="attention")
ioi.add_edge("attn.7.h3", "mlp.8", weight=0.50, edge_type="direct")
ioi.add_edge("attn.8.h6", "mlp.8", weight=0.47, edge_type="direct")
ioi.add_edge("mlp.8", "attn.9.h9", weight=0.82, edge_type="direct")
ioi.add_edge("mlp.8", "attn.10.h0", weight=0.75, edge_type="direct")
ioi.add_edge("attn.7.h3", "mlp.9", weight=0.38, edge_type="direct")
ioi.add_edge("mlp.9", "attn.9.h9", weight=0.55, edge_type="direct")
ioi.add_edge("mlp.9", "attn.10.h0", weight=0.48, edge_type="direct")

# Visualize using .visualize()
ioi.visualize(layout="layer", figsize=(14, 8), node_size=1000, font_size=8)
print(ioi.summary())

The IOI circuit reveals a clean **three-stage pipeline**: duplicate-token heads detect the repeated name, S-inhibition heads suppress it, and name-mover heads copy the *other* name to the output. Just 9 nodes and 12 edges out of GPT-2's full 144-head network are enough to explain the behavior.

---
## 6. Steering Vectors — Pushing GPT-2's Sentiment

A steering vector is the difference in mean activations between positive and negative examples at a chosen layer. By adding scaled copies of this vector during inference, we can shift the model's internal representation toward positive or negative sentiment without retraining.

In [ ]:
from glassboxllms.primitives.probes.activation_store import ActivationStore
from glassboxllms.visualization.plots import plot_steering_effects

# --- 1. Compute steering vector at the best probing layer ---
steer_store = ActivationStore(model)
target_layer = "h.10"  # late layer where sentiment is linearly encoded

pos_texts = [
    "I absolutely love this movie, it was fantastic!",
    "This is the best day of my life, I'm so happy!",
    "What a wonderful experience, highly recommended!",
    "The food was delicious and the service was excellent.",
    "Amazing performance, the actors were brilliant!",
]
neg_texts = [
    "I hate this movie, it was terrible and boring.",
    "This is the worst day ever, everything went wrong.",
    "What a horrible experience, never going back!",
    "The food was disgusting and the service was awful.",
    "Terrible performance, the actors were embarrassing.",
]

pos_acts = steer_store.extract(pos_texts, tokenizer, layers=[target_layer], pooling="mean")[target_layer]
neg_acts = steer_store.extract(neg_texts, tokenizer, layers=[target_layer], pooling="mean")[target_layer]

steering_vector = pos_acts.mean(axis=0) - neg_acts.mean(axis=0)
print(f"Steering vector at {target_layer}: norm={np.linalg.norm(steering_vector):.2f}, dims={steering_vector.shape[0]}")
print(f"Top-5 dimensions: {np.argsort(np.abs(steering_vector))[-5:][::-1]}")

# --- 2. Measure effect: project a neutral sentence onto the vector at each scale ---
neutral = "The movie was okay and had some interesting parts."
neutral_act = steer_store.extract([neutral], tokenizer, layers=[target_layer], pooling="mean")[target_layer][0]

sv_unit = steering_vector / np.linalg.norm(steering_vector)
base_proj = float(np.dot(neutral_act, sv_unit))

effects = {}
for alpha in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    steered = neutral_act + alpha * steering_vector
    proj = float(np.dot(steered, sv_unit))
    effects[f"{alpha:+.0f}x"] = {"sentiment_projection": proj, "shift": proj - base_proj}

print("\nSteering effects (sentiment projection):")
for k, v in effects.items():
    print(f"  {k}: projection={v['sentiment_projection']:.2f}  shift={v['shift']:+.2f}")

# --- 3. Visualize ---
fig = plot_steering_effects(effects, metric="sentiment_projection")
fig.suptitle(f"Steering Vector Effect on Sentiment ({target_layer})", fontsize=14, y=1.02)
plt.show()

The sentiment projection scales linearly with steering strength, confirming the vector captures a clean linear direction. This is the same direction the probe learned to read in Section 1 — steering writes to it, probing reads from it.

---
## 7. Causal Patching — Which Layers Drive Name Prediction?

Causal patching swaps a layer's activation from a clean run into a corrupted run. If the answer shifts back toward the clean answer, that layer causally carries the signal. We test GPT-2's Indirect Object Identification: *"When Mary and John went to the store, John gave a drink to ___"* (answer: Mary).

In [ ]:
import torch.nn.functional as F
from operator import attrgetter

source_prompt = "When Mary and John went to the store, John gave a drink to"
corrupted_prompt = "When Mary and John went to the store, Mary gave a drink to"
target_token = " Mary"
target_id = tokenizer.encode(target_token)[-1]

corr_ids = tokenizer(corrupted_prompt, return_tensors="pt").input_ids
clean_ids = tokenizer(source_prompt, return_tensors="pt").input_ids

with torch.no_grad():
    baseline_p = F.softmax(lm_model(corr_ids).logits[0, -1], dim=-1)[target_id].item()
    clean_p = F.softmax(lm_model(clean_ids).logits[0, -1], dim=-1)[target_id].item()
print(f'P("{target_token.strip()}") clean={clean_p:.3f}, corrupted={baseline_p:.3f}\n')

# Patch each MLP and measure how much P(Mary) is restored
effects = []
for i in range(model.config.n_layer):
    mod = attrgetter(f"transformer.h.{i}.mlp")(lm_model)
    cache = {}
    def cache_hook(module, input, output, c=cache):
        c['act'] = output.detach().clone()
        return output
    h = mod.register_forward_hook(cache_hook)
    with torch.no_grad(): lm_model(clean_ids)
    h.remove()

    def patch_hook(module, input, output, c=cache):
        return c['act']
    h = mod.register_forward_hook(patch_hook)
    with torch.no_grad(): patched_logits = lm_model(corr_ids).logits
    h.remove()

    patched_p = F.softmax(patched_logits[0, -1], dim=-1)[target_id].item()
    ie = patched_p - baseline_p
    effects.append(ie)
    print(f"  mlp.{i}: P(Mary)={patched_p:.3f}, IE={ie:+.4f}")

print(f"\nStrongest MLP: layer {np.argmax(np.abs(effects))} (IE = {effects[np.argmax(np.abs(effects))]:+.4f})")

In [ ]:
# Bar chart of MLP indirect effects across layers
fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#c44e52" if e > 0 else "#4c72b0" for e in effects]
ax.bar(range(len(effects)), effects, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Layer")
ax.set_ylabel("Indirect Effect (Δ P(Mary))")
ax.set_title('Causal Patching: Which MLP layers recover P(" Mary") in the IOI task?')
ax.set_xticks(range(len(effects)))
ax.set_xticklabels([str(i) for i in range(len(effects))])
ax.axhline(0, color="black", linewidth=0.8)
fig.tight_layout()
plt.show()

MLP 0 shows the strongest causal effect — it writes the name information into the residual stream early. Later MLPs have small positive or negative contributions, showing the distributed nature of the computation.

---
## 8. Attention Patterns — What Does GPT-2 Focus On?

Attention heatmaps reveal which tokens each head attends to when processing a sentence. By comparing an early-layer head (layer 0) with a late-layer head (layer 11), we can see how attention shifts from local, positional patterns to long-range, semantic dependencies.

In [ ]:
from glassboxllms.visualization.plots import plot_attention_heatmap

text = "The cat sat on the mat because it was tired"
inputs = tokenizer(text, return_tensors="pt")
tokens = [tokenizer.decode(t) for t in inputs["input_ids"][0]]

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# outputs.attentions is a tuple of (batch, heads, seq, seq) per layer
# Extract head 0 from an early layer and a late layer
attn_early = outputs.attentions[0][0, 0].numpy()   # layer 0, head 0
attn_late = outputs.attentions[11][0, 0].numpy()    # layer 11, head 0

# Side-by-side heatmaps
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, attn, title in [
    (axes[0], attn_early, "Layer 0, Head 0 (early)"),
    (axes[1], attn_late, "Layer 11, Head 0 (late)"),
]:
    sns.heatmap(attn, ax=ax, cmap="viridis", xticklabels=tokens,
                yticklabels=tokens, square=True, vmin=0, vmax=attn.max())
    ax.set_title(title)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

fig.suptitle("Attention Patterns: Early vs Late Layer (GPT-2)", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# Individual detailed view using the library helper
fig_detail = plot_attention_heatmap(attn_late, tokens, layer=11, head=0)
plt.show()

The early-layer head shows a strong diagonal pattern (each token attending to itself or its immediate neighbor), while the late-layer head spreads attention across semantically relevant tokens — for example, "it" attending back to "cat". This confirms that GPT-2 builds local context first and resolves long-range dependencies in deeper layers.

---
## 9. Animated Visualizations (Manim)

The glassbox-llms visualization module includes cinematic Manim animations that bring interpretability concepts to life. Below are pre-rendered animations using real data from the experiments above.

In [ ]:
from IPython.display import Video, display

# Probing: scatter with decision boundary
print("Probing Hyperplane — Sentiment classification in activation space")
try:
    display(Video("../media/demo/ProbingHyperplaneScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

In [ ]:
# Circuit Discovery: Animated IOI circuit graph
print("Circuit Discovery — GPT-2 IOI Circuit")
try:
    display(Video("../media/demo/CircuitDiscoveryScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

In [ ]:
# Steering: Before/after activation shift
print("Steering Vector — Sentiment direction in activation space")
try:
    display(Video("../media/demo/SteeringVectorScene.mp4", embed=True, width=800))
except FileNotFoundError:
    print("  (Pre-render with: python scripts/render_demo_animations.py)")

These animations are generated from the same real GPT-2 experiment data shown in the static plots above, converted through the `glassboxllms.visualization.adapters` layer into Manim scene data.